In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[4]")
    .appName("TP-RDD")
    .getOrCreate()
)

sc = spark.sparkContext
sc.setLogLevel("WARN")

DATA = "/home/jovyan/work"

In [2]:
print("Mode d'exécution :", sc.master)
print("Parallélisme par défaut :", sc.defaultParallelism)
print("Version :", sc.version)

Mode d'exécution : local[4]
Parallélisme par défaut : 4
Version : 4.0.0


In [3]:
print(sc.uiWebUrl)

http://ee2194785683:4041


In [4]:
citations = [
    "Spark est un moteur de calcul distribue",
    "Spark traite les donnees en memoire",
    "Hadoop stocke les donnees sur disque",
    "Spark est rapide et Spark est facile",
    "Les RDD sont immuables et distribues",
    "Le calcul distribue repose sur les RDD",
]

with open(DATA + "/citations.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(citations) + "\n")

print("Fichier citations.txt créé.")

Fichier citations.txt créé.


In [6]:
x = sc.parallelize(range(1, 21), 4)

print("Nombre de partitions :", x.getNumPartitions())
print("Contenu des partitions :", x.glom().collect())

Nombre de partitions : 4
Contenu des partitions : [[1, 2, 3, 4, 5], [6, 7, 8, 9, 10], [11, 12, 13, 14, 15], [16, 17, 18, 19, 20]]


In [7]:
x_auto = sc.parallelize(range(1, 21))

print("Partitions automatiques :", x_auto.getNumPartitions())
print("Parallélisme par défaut :", sc.defaultParallelism)

Partitions automatiques : 4
Parallélisme par défaut : 4


In [8]:
txt = sc.textFile(DATA + "/citations.txt", 2)

print("Nombre de lignes :", txt.count())
print("Première ligne :", txt.first())
print("Type :", type(txt))

Nombre de lignes : 6
Première ligne : Spark est un moteur de calcul distribue
Type : <class 'pyspark.core.rdd.RDD'>


In [9]:
inexistant = sc.textFile(DATA + "/fichier_qui_n_existe_pas.txt")
print("RDD défini")

RDD défini


In [11]:
with open(DATA + "/fichier_qui_n_existe_pas.txt", "w", encoding="utf-8") as f:
    f.write("Bonjour Spark\n")

In [12]:
inexistant.count()

1

In [13]:
nombres = sc.parallelize([7, 10, 12, 17, 19], 2)

carres = nombres.map(lambda a: a ** 2)

print("Carrés :", carres.collect())
print("Nombre initial :", nombres.count())
print("Nombre après map :", carres.count())

Carrés : [49, 100, 144, 289, 361]
Nombre initial : 5
Nombre après map : 5


In [14]:
couples = nombres.map(lambda a: (a // 2, a % 2))

print(couples.collect())

[(3, 1), (5, 0), (6, 0), (8, 1), (9, 1)]


In [15]:
pairs = nombres.filter(lambda a: a % 2 == 0)
tests = nombres.map(lambda a: a % 2 == 0)

print("Avec filter :", pairs.collect())
print("Avec map :", tests.collect())

Avec filter : [10, 12]
Avec map : [False, True, True, False, False]


In [16]:
groupes = sc.parallelize([(1, 2), (3, 5), (8, 10)])

plat = groupes.flatMap(lambda couple: couple)
avec_map = groupes.map(lambda couple: couple)

print("Avec flatMap :", plat.collect())
print("Avec map :", avec_map.collect())

Avec flatMap : [1, 2, 3, 5, 8, 10]
Avec map : [(1, 2), (3, 5), (8, 10)]


In [17]:
pairs_flatmap = nombres.flatMap(
    lambda a: [a] if a % 2 == 0 else []
)

print(pairs_flatmap.collect())

[10, 12]


In [18]:
z = sc.parallelize([1, 1, 2, 3, 4, 5, 5, 5, 6, 6])

uniques = z.distinct()

print(uniques.collect())

[4, 1, 5, 2, 6, 3]


In [19]:
y = sc.parallelize([6, 8, 9, 1, 2, 3], 3)

print("count :", y.count())
print("first :", y.first())
print("take(3) :", y.take(3))
print("take(100) :", y.take(100))
print("takeOrdered(3) :", y.takeOrdered(3))
print("collect :", y.collect())

count : 6
first : 6
take(3) : [6, 8, 9]
take(100) : [6, 8, 9, 1, 2, 3]
takeOrdered(3) : [1, 2, 3]
collect : [6, 8, 9, 1, 2, 3]


In [20]:
y = sc.parallelize([6, 8, 9, 1, 2, 3], 3)

print("count :", y.count())
print("first :", y.first())
print("take(3) :", y.take(3))
print("take(100) :", y.take(100))
print("takeOrdered(3) :", y.takeOrdered(3))
print("collect :", y.collect())

count : 6
first : 6
take(3) : [6, 8, 9]
take(100) : [6, 8, 9, 1, 2, 3]
takeOrdered(3) : [1, 2, 3]
collect : [6, 8, 9, 1, 2, 3]


In [21]:
print("Somme avec reduce :", y.reduce(lambda a, b: a + b))
print("Somme avec sum :", y.sum())
print("Maximum :", y.max())
print("Minimum :", y.min())
print("Moyenne :", y.mean())

Somme avec reduce : 29
Somme avec sum : 29
Maximum : 9
Minimum : 1
Moyenne : 4.833333333333333


In [22]:
vide = sc.parallelize([])

print(vide.take(3))

[]


In [24]:
if vide.isEmpty():
    print("Le RDD est vide.")
else:
    print(vide.first())

Le RDD est vide.


In [25]:
print(y.take(3))

[6, 8, 9]


In [26]:
mots = txt.flatMap(lambda ligne: ligne.lower().split(" "))

print("Nombre total de mots :", mots.count())
print("Nombre de mots distincts :", mots.distinct().count())

Nombre total de mots : 39
Nombre de mots distincts : 25


In [27]:
comptage = (
    mots
    .map(lambda mot: (mot, 1))
    .reduceByKey(lambda a, b: a + b)
)

print(comptage.collect())

[('est', 3), ('un', 1), ('distribue', 2), ('traite', 1), ('les', 4), ('donnees', 2), ('hadoop', 1), ('sur', 2), ('rapide', 1), ('et', 2), ('rdd', 2), ('sont', 1), ('immuables', 1), ('repose', 1), ('spark', 4), ('moteur', 1), ('de', 1), ('calcul', 2), ('en', 1), ('memoire', 1), ('stocke', 1), ('disque', 1), ('facile', 1), ('distribues', 1), ('le', 1)]


In [28]:
top5 = comptage.takeOrdered(5, key=lambda couple: -couple[1])

print(top5)

[('les', 4), ('spark', 4), ('est', 3), ('distribue', 2), ('donnees', 2)]


In [29]:
lignes_spark = txt.filter(
    lambda ligne: "Spark" in ligne.split(" ")
)

longueurs = txt.map(lambda ligne: len(ligne.split(" ")))

print("Lignes contenant Spark :", lignes_spark.count())
print("Longueur maximale :", longueurs.max())

Lignes contenant Spark : 3
Longueur maximale : 7


In [30]:
comptage.coalesce(1).saveAsTextFile(DATA + "/wc_out")

In [31]:
import os

print("Fichiers créés :", os.listdir(DATA + "/wc_out"))

resultat = sc.textFile(DATA + "/wc_out")

print("Nombre de lignes :", resultat.count())
print("Exemples :", resultat.take(5))

Fichiers créés : ['.part-00000.crc', '._SUCCESS.crc', 'part-00000', '_SUCCESS']
Nombre de lignes : 25
Exemples : ["('est', 3)", "('un', 1)", "('distribue', 2)", "('traite', 1)", "('les', 4)"]


In [32]:
comptage.coalesce(1).saveAsTextFile(DATA + "/wc_out_2")

### Q5.1 — Transformations et actions

Les transformations retournent un nouveau RDD :
map, filter, flatMap, distinct, reduceByKey.

Les actions déclenchent le calcul pour obtenir un résultat ou écrire des données :
count, first, take, reduce, collect, takeOrdered, saveAsTextFile.

Le critère de distinction est qu’une transformation définit un nouveau RDD,
tandis qu’une action demande l’exécution du traitement.

### Q5.2 — Jobs Spark

J’observe ___ jobs dans l’interface Spark.

Les actions, comme count() et collect(), déclenchent les calculs
et peuvent lancer un ou plusieurs jobs.
Une transformation seule, comme map() ou filter(), ne lance pas de job.

### Q5.3 — Définition d’un RDD

Un RDD (Resilient Distributed Dataset) est une collection immuable
divisée en partitions que Spark peut traiter en parallèle.

Spark conserve les dépendances permettant de recalculer les partitions
perdues si les données nécessaires restent disponibles.